In [ ]:
%%capture
!pip install gdown
!pip install bertopic

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from bertopic import BERTopic
from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import LabelEncoder

# Step 1: Load dataset

In [ ]:
# News Groups dataset
newsgroups_data = pd.read_csv('data/fetch_7newsgroups.csv')
newsgroups_data.dropna(inplace=True,subset='label')
label_encoder_newsgroups = LabelEncoder()
newsgroups_data['topic'] = label_encoder_newsgroups.fit_transform(newsgroups_data['label'])

newsgroup_text = newsgroups_data['text'].tolist()
newsgroup_labels = newsgroups_data['topic'].tolist()


# BBC News dataset
bbc_news = pd.read_csv("data/bbc-news-data.csv",sep="\t")  # Load the BBC dataset

label_encoder_bbc = LabelEncoder()
bbc_news['topic'] = label_encoder_bbc.fit_transform(bbc_news['category'])

bbc_texts = bbc_news.apply(lambda r:r.title+r.content,axis=1).to_list()
bbc_labels = bbc_news['topic'].to_list()

In [ ]:
import re
import string


def preprocess_text(text,preprocess=True):
    if not preprocess:
        return text
    # Convert text to lowercase
    text = text.lower()

    # Replace email addresses with 'EMAIL'
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', 'EMAIL', text)

    # Replace URLs with 'HTTP'
    text = re.sub(r'https?://\S+|www\.\S+', 'HTTP', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans(' ', ' ', string.punctuation))
    text = ' '.join(text.split())# drop white spaces
    return text


In [ ]:
dataset_name = 'newsgroup'

if dataset_name == 'bbc':
    documents = bbc_texts
    true_labels = bbc_labels
    test_ratio = 0.2
    do_preprocess = False
elif dataset_name == 'newsgroup':
    documents = newsgroup_text
    true_labels = newsgroup_labels
    test_ratio = 0.05
    do_preprocess = True


In [ ]:
documents = [preprocess_text(doc,do_preprocess) for doc in documents]


In [ ]:
# Step 2: Split the dataset 80-20 into train and test sets
train_docs, test_docs, train_labels, test_labels = train_test_split(documents, true_labels, test_size=test_ratio, random_state=42)


# **Topic Modeling with BERTopic**
<img src="https://raw.githubusercontent.com/MaartenGr/BERTopic/master/images/logo.png" width="20%">



## BERTopic
BERTopic is a topic modeling technique that leverages ?? transformers and a custom class-based TF-IDF to create dense clusters allowing for easily interpretable topics whilst keeping important words in the topic descriptions.

<br>



In [ ]:
from sentence_transformers import SentenceTransformer

# List of sentence transformers
sentence_transformers = [
    "all-mpnet-base-v2",  # Good general-purpose model
    "multi-qa-mpnet-base-dot-v1" #Strong performance on various tasks
]
transformer = sentence_transformers[1]

In [ ]:
%%time
start_time = time.time()

topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True, embedding_model=transformer)
topics,probs = topic_model.fit_transform(train_docs)

end_time = time.time()
training_time = end_time - start_time



In [ ]:
freq = topic_model.get_topic_info(); freq.head(7)

# **Visualization**

In [ ]:
# topic_model.visualize_topics()

In [ ]:
# topic_model.visualize_distribution(probs[200], min_probability=0.005)

In [ ]:
topic_model.visualize_hierarchy(top_n_topics=50)

In [ ]:
# topic_model.visualize_barchart(top_n_topics=8)

In [ ]:
# Save model
topic_model.save(f"model/my_model_{dataset_name}_{transformer}")

In [ ]:
# Load model
# my_model = BERTopic.load(f"models/my_model_{dataset_name}_{transformer}")

In [ ]:
%%time
start_time = time.time()

test_topics, test_probs = topic_model.transform(test_docs)

end_time = time.time()
testing_time = end_time - start_time

In [ ]:
# Step 5: Evaluate the model using Purity and NMI on test data

# 5.1 Helper function to compute purity
def calculate_purity(topics, true_labels):
    df = pd.DataFrame({'Topic': topics, 'True Label': true_labels})
    majority_count = 0
    for topic, group in df.groupby('Topic'):
        majority_class = group['True Label'].mode()[0]
        majority_count += (group['True Label'] == majority_class).sum()
    purity = majority_count / len(df)
    return purity

# Step 6: Compute Purity and NMI for test data
test_purity = calculate_purity(test_topics, test_labels)
test_nmi = normalized_mutual_info_score(test_labels, test_topics)

# Step 7: Output results
print(f"Test Purity: {test_purity}")
print(f"Test NMI: {test_nmi}")



In [ ]:
from scipy import stats


# Step 1: Create a DataFrame for training data
train_df = pd.DataFrame({
    'topic': topics,
    'label': train_labels
})

# Step 2: Group by 'topic' and count how many of each label there are for each topic in the training data
train_topic_label_counts = train_df.groupby(['topic', 'label']).size().unstack(fill_value=0)

# Step 3: Assign the mode (most frequent label) for each topic in the training data
topic_to_mode_label = train_df.groupby('topic')['label'].agg(lambda x: stats.mode(x)[0])

# Step 4: Map the predicted test topics to actual labels using the mapping from training data
mapped_test_labels = [topic_to_mode_label.get(topic, None) for topic in test_topics]

# Create a DataFrame to combine mapped test labels and actual test labels
test_results_df = pd.DataFrame({
    'predicted_label': mapped_test_labels,
    'true_label': test_labels
})



In [ ]:
from sklearn.metrics import classification_report
print(classification_report(test_results_df.true_label,test_results_df.predicted_label,digits=3))

In [ ]:
# Generate classification report
report = classification_report(test_results_df.predicted_label,test_results_df.true_label,digits=3, output_dict=True)

report['purity'] = test_purity
report['nmi'] = test_nmi
report['training_time'] = training_time
report['testing_time'] = testing_time
# Convert the report to a DataFrame
report_df = pd.DataFrame(report).transpose()

# Specify the file path and sheet name
file_path = 'Topic_modeling_classification_report.xlsx'
sheet_name = f'{dataset_name}_{transformer}'

# Write the DataFrame to an Excel sheet
with pd.ExcelWriter(file_path, engine='openpyxl',  mode='a', if_sheet_exists='new') as writer:
    report_df.to_excel(writer, sheet_name=sheet_name)

print(f"Classification report saved to '{file_path}' in sheet '{sheet_name}'")
